# Dimensionality Reduction - PCA + t-SNE


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/13-Dimensionality-Reduction-PCA-tSNE.ipynb)

We previously learned how to use the SVD as a tool for constructing low-rank matrices.

We now consider it as a tool for transforming (i.e., reducing the dimension of) our data.

## Overview

Collected data is often high-dimensional, and a large number of features is challenging to work with.

![The blind men and the elephant: each one measures a few dimensions of the same object and reaches a different conclusion. Dimensionality reduction looks for the few coordinates that describe the whole. (Illustration of the traditional parable; artist unknown.)](figs/elephant_perspective.png)

## High-Dimensional Challenges

We have seen some of these challenges already, in particular:

- the curse of dimensionality, where data points become sparse in higher dimensions and distance metrics have less meaning,
- overfitting, where high-dimensional data can lead to overly complex models that fit to noise in the data as opposed to the actual signal,
- computational complexity, high-dimensional data requires more computing power and memory,
- visualization, where high-dimensional data makes understanding and interpreting the data difficult.

---

How can we reduce the dimension of our data but still preserve the most important information in our dataset?

We consider two techniques:

- Principal Component Analysis (PCA)
- t-distributed stochastic neighbor embedding (t-SNE)


We will demonstrate the relationship between PCA and the SVD.

t-SNE is an alternative nonlinear method for dimensionality reduction.


# PCA

## Dimensionality Reduction

Input: $\mathbf{x}_1,\ldots, \mathbf{x}_m$ with  $\mathbf{x}_i \in \mathbb{R}^n \: \: \forall \: i \in \{1, \ldots, m\}.$

Output: $\mathbf{y}_1,\dots, \mathbf{y}_m$  with  $\mathbf{y}_i \in \mathbb{R}^d \: \: \forall \: i \in \{1, \dots, m\}$.

The goal is to compute the new data points $\mathbf{y}_i$ such that <font color="red"> $d << n$ </font> while still preserving the most information contained in the data points $\mathbf{x}_i$.

Be aware that row i of the data matrix $X_0$ is the $n$ dimensional vector $\mathbf{x}_i$. This keeps our matrix in the structure $m$ data rows and $n$ features (columns). The same is true for $Y$ (i.e., there are $m$ rows with $d$ features). Throughout this lecture $m$ is the number of points and $n$ the number of features -- the same matrix-theory convention as last lecture's SVD.

$$
X_0 =
\begin{bmatrix}
x_{11} & x_{12} & \dots & x_{1n} \\
x_{21} & x_{22} & \dots & x_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
x_{m1} & x_{m2} & \dots & x_{mn}
\end{bmatrix} \:
\xrightarrow[\text{PCA}]{} \:
Y = \begin{bmatrix}
y_{11} & y_{12} & \dots & y_{1d} \\
y_{21} & y_{22} & \dots & y_{2d} \\
\vdots & \vdots & \ddots & \vdots \\
y_{m1} & y_{m2} & \dots & y_{md}
\end{bmatrix}
$$

## PCA Application: Genes Mirror Geography

We consider a dataset from @novembre2008genes. The authors collected DNA data from about 3000 individuals of European ancestry.

Each feature is a SNP (single nucleotide polymorphism): a position in the genome where the base pair varies from person to person. At each SNP, a person's genotype is coded by how many copies of the less common (minor) variant they carry -- 0, 1, or 2, one copy from each parent.

The data for each individual consisted of approximately 500k SNPs. This means the data matrix we are working with is roughly 3000 x 500k, with entries in $\{0, 1, 2\}$.

---

The authors performed PCA and plotted the 1387 individuals whose four grandparents all came from the same country, so that each point can be colored by a single country of origin.

![Image Credit @novembre2008genes](figs/PCA_genes_Europe.png)

For comparison, a color coded map of western Europe is added and the same color coding was applied to the data samples by country of origin.

---

Key observations:

- the first principal components of the data almost reproduce the map of Europe, i.e., they appear to correspond to latitude and longitude.
- SNPs are similar geographically
- DNA of an individual reveals their birthplace within a few hundred kilometers


Would a similar study in the USA be effective?


**Would it work in the USA?**

Much less well. The European pattern exists because, for most of history, people married within a few tens of kilometers of where they were born, so genetic similarity accumulated along geography. The US population is a few centuries of migration from several continents, and ancestry is only loosely tied to birthplace. A PCA of US genomes mostly separates continental ancestry (European, African, East Asian, Indigenous American), not states; birthplace to within a few hundred kilometers is out of reach.


## PCA Overview

The following describes the process to perform PCA and obtain your reduced data set.

The goal is given:

Input: $X_0\in\mathbb{R}^{m\times n}$, produce

Output: $Y\in\mathbb{R}^{m\times d}$ with $d << n$.

## PCA Step 1: Center the Data

The first step is to center the data (subtract the mean of each column): $X_0 \rightarrow X$.

Example: three points ($m = 3$) with two features ($n = 2$):
$$
X_0 =
\begin{bmatrix}
12 & 6\\
9 & 6\\
9 & 3 \end{bmatrix}
$$

The mean, per column, across rows is:
$$
\boldsymbol{\mu} = \begin{bmatrix} 10 & 5 \end{bmatrix}
$$

The mean-centered dataset is

$$
X = \begin{bmatrix}
2 & 1\\
-1 & 1\\
-1 & -2 \end{bmatrix}
$$

We will carry this example through the whole PCA recipe.

The next step is to determine the directions of the data that correspond to the largest variances. These are the principal components.

## Least Squares Interpretation

Centered data often clusters along a line (or other low-dimensional subspace of $\mathbb{R}^{n}$).

![](figs/PCA_variance.png)

The sum of **variances** (squared distances to the mean) of the projected points is a **maximum**.

![](figs/PCA_residual.png)

The sum of **residuals** (squared distances from the points to the line) is a **minimum**.


What is the statistical entity that measures the variability in the data?


Answer: the covariance matrix.


## Covariance Matrix

Let $X\in\mathbb{R}^{m\times n}$ contain the centered data. The sample covariance matrix for zero-mean data is:

$$
S = \frac{1}{m-1}X^{T}X
$$


Example:
$$
X = \begin{bmatrix}
2 & 1\\
-1 & 1\\
-1 & -2 \end{bmatrix}
$$

$$
S = \frac{1}{2}
\begin{bmatrix}
2 & -1 & -1\\
1 & 1 & -2 \end{bmatrix}
\begin{bmatrix}
2 & 1\\
-1 & 1\\
-1 & -2 \end{bmatrix}
= \frac{1}{2}
\begin{bmatrix}
6 & 3\\
3 & 6 \end{bmatrix}
=
\begin{bmatrix}
3 & 1.5\\
1.5 & 3 \end{bmatrix}
$$

The matrix $S$ is symmetric, i.e., $S = S^{T}.$ Its diagonal holds the variance of each feature (3 and 3) and its off-diagonal the covariance between them (1.5).


## Understanding the Formula

The formula is:
$$S = \frac{1}{m-1}X^{T}X$$

where $X\in\mathbb{R}^{m\times n}$ contains **centered data**.

#### Key Setup

First, let's understand what $X$ represents:

- **$m$ rows** = number of observations/samples
- **$n$ columns** = number of features/variables
- **Centered data** means each column has mean 0 (we've subtracted the column mean from each entry)

So if we denote the columns of $X$ as vectors, we have:
$$X = \begin{bmatrix} | & | & & | \\ \mathbf{x}_1 & \mathbf{x}_2 & \cdots & \mathbf{x}_n \\ | & | & & | \end{bmatrix}$$

where each $\mathbf{x}_j \in \mathbb{R}^m$ is a centered column vector (mean = 0).

#### What Does $X^T X$ Compute?

When we compute $X^T X$:

- Dimensions: $(n \times m) \cdot (m \times n) = n \times n$
- The $(i,j)$ entry of $X^T X$ is: $(\mathbf{x}_i)^T \mathbf{x}_j = \sum_{k=1}^{m} x_{ki} \cdot x_{kj}$

This is the **dot product** between column $i$ and column $j$ of $X$.

#### Why This Gives Covariance

The sample covariance between features $i$ and $j$ is defined as:
$$\text{Cov}(i,j) = \frac{1}{m-1}\sum_{k=1}^{m}(x_{ki} - \bar{x}_i)(x_{kj} - \bar{x}_j)$$

But since the data is **centered**, we have $\bar{x}_i = 0$ and $\bar{x}_j = 0$, so:
$$\text{Cov}(i,j) = \frac{1}{m-1}\sum_{k=1}^{m}x_{ki} \cdot x_{kj} = \frac{1}{m-1}(\mathbf{x}_i)^T \mathbf{x}_j$$

This is exactly the $(i,j)$ entry of $\frac{1}{m-1}X^T X$!

#### The Result

Therefore, $S = \frac{1}{m-1}X^{T}X$ is an $n \times n$ matrix where:

- **Diagonal entries** $S_{ii}$ = variance of feature $i$
- **Off-diagonal entries** $S_{ij}$ = covariance between features $i$ and $j$

This is precisely the **sample covariance matrix**.

**Note:** The factor of $(m-1)$ instead of $m$ gives us the unbiased sample estimator (Bessel's correction).


## Variance Along a Direction

Pick any direction: a unit vector $\mathbf{w}\in\mathbb{R}^n$ with $\|\mathbf{w}\| = 1$. Projecting every centered point onto it gives one coordinate per point, the vector $X\mathbf{w}\in\mathbb{R}^m$, which has mean zero because the columns of $X$ do.

Its variance is

$$
\operatorname{Var}(X\mathbf{w}) = \frac{1}{m-1}(X\mathbf{w})^{T}(X\mathbf{w}) = \mathbf{w}^{T}\left(\frac{1}{m-1}X^{T}X\right)\mathbf{w} = \mathbf{w}^{T}S\mathbf{w}.
$$

So "find the direction of maximum variance" means: maximize $\mathbf{w}^{T}S\mathbf{w}$ subject to $\mathbf{w}^{T}\mathbf{w} = 1$.

In the example, $\mathbf{w} = \begin{bmatrix}1 & 0\end{bmatrix}^T$ (feature 1 alone) gives $\mathbf{w}^TS\mathbf{w} = 3$, while $\mathbf{w} = \frac{1}{\sqrt{2}}\begin{bmatrix}1 & 1\end{bmatrix}^T$ gives $\frac{1}{2}(3 + 1.5 + 1.5 + 3) = 4.5$ and $\frac{1}{\sqrt{2}}\begin{bmatrix}1 & -1\end{bmatrix}^T$ gives $1.5$.


---

Setting the gradient of the Lagrangian $\mathbf{w}^{T}S\mathbf{w} - \lambda(\mathbf{w}^{T}\mathbf{w} - 1)$ to zero gives

$$
2S\mathbf{w} - 2\lambda\mathbf{w} = 0 \quad\Longrightarrow\quad S\mathbf{w} = \lambda\mathbf{w}.
$$

- The maximizer is an **eigenvector** of $S$, and the variance it achieves is $\mathbf{w}^{T}S\mathbf{w} = \lambda\,\mathbf{w}^{T}\mathbf{w} = \lambda$: its **eigenvalue**.
- The largest variance therefore belongs to the eigenvector with the largest eigenvalue. The next direction, constrained to be orthogonal to the first, is the eigenvector with the second-largest eigenvalue, and so on.
- The two pictures above are the same problem. For each point, Pythagoras gives $\|\mathbf{x}_i\|^2 = (\mathbf{x}_i\cdot\mathbf{w})^2 + (\text{distance from } \mathbf{x}_i \text{ to the line})^2$. Summed over the points, the left side does not depend on $\mathbf{w}$, so maximizing the projected variance *is* minimizing the squared residuals.


This is why the next step is an eigendecomposition of $S$.


## Spectral Decomposition

We use the fact that the covariance matrix $S$ is symmetric to apply the Spectral Decomposition, which states:

> Every real symmetric matrix $S$ has the factorization $V\Lambda V^{T}$, where $\Lambda$ is a diagonal matrix that contains the eigenvalues of S and the columns of $V$ are orthonormal eigenvectors of $S$.

This is a special case of eigendecomposition for symmetric matrices.

You can refresh your memory about this in the [Linear Algebra Refresher](A7-Linear-Algebra-Refresher.qmd#eigendecomposition).


---

The covariance matrix $S \in \mathbb{R}^{n\times n}$ has the spectral decomposition $S = V\Lambda V^T$ where

$$
\Lambda =
\begin{bmatrix}
\lambda_1                                   \\
& \lambda_2             &   & \text{\Large0}\\
&               & \ddots                \\
& \text{\Large0} &   & \lambda_{n-1}            \\
&               &   &   & \lambda_n
\end{bmatrix}
$$
with $\lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_{n-1} \geq \lambda_n$, and
$$
V = \begin{bmatrix}
\bigg| & \bigg| &  & \bigg| & \bigg| \\
\mathbf{v}_1   & \mathbf{v}_2  & \dots & \mathbf{v}_{n-1}   & \mathbf{v}_n  \\
\bigg| & \bigg| & & \bigg| & \bigg|
\end{bmatrix}
$$

with $S\mathbf{v}_i = \lambda_i \mathbf{v}_i$, $\|\mathbf{v}_i\| = 1$ and $\mathbf{v}_i \perp \mathbf{v}_j$ for $i\neq j$.

---

The previous decomposition was for all $n$ dimensions. To obtain our reduced data, we take the first $d$ columns of $V$, i.e.,

$$
V' = \begin{bmatrix}
\bigg| & \bigg| &  & \bigg|   \\
\mathbf{v}_1   & \mathbf{v}_2  & \dots & \mathbf{v}_{d}  \\
\bigg| & \bigg| & & \bigg|
\end{bmatrix},
$$
and the $d\times d$ upper block of matrix $\Lambda$
$$
\Lambda' =
\begin{bmatrix}
\lambda_1  &  & \\
  & \lambda_2  &   \\
&               & \ddots                \\
&               &   &   & \lambda_d
\end{bmatrix}.
$$

## PCA Interpretation

* The direction $\mathbf{v}_i$ is the $i$-th principal component and the corresponding $\lambda_i$ is the variance of the data along it -- the $i$-th largest variance in the dataset, by the argument above.

  * In other words, $\mathbf{v}_1$ is the first principal component and is the direction that accounts for the most variance in the dataset.

  * The vector $\mathbf{v}_d$ is the $d$-th principal component and is the direction that accounts for the $d$-th most variance in the dataset.

The reduced data matrix is obtained by computing

$$
Y = XV'.
$$

Its $j$-th column is $X\mathbf{v}_j$, so $\operatorname{Var}(Y_j) = \mathbf{v}_j^TS\mathbf{v}_j = \lambda_j$: the columns of $Y$ have exactly the variances $\lambda_1, \dots, \lambda_d$, and they are uncorrelated with one another.


## Spatial Interpretation -- Dimensions

The operation $Y = XV'$ performs a **change of basis** and **projection** of your data:

- $X \in \mathbb{R}^{m \times n}$: each of the $m$ rows is a data point in the original $n$-dimensional feature space
- $V' \in \mathbb{R}^{n \times d}$: contains the first $d$ principal components as columns (these are orthonormal vectors)
- $Y \in \mathbb{R}^{m \times d}$: each row is now a data point in the new $d$-dimensional space

## Geometric Interpretation

1. **Rotation**: The multiplication by $V'$ rotates your coordinate system. The columns of $V'$ define a new set of $d$ orthogonal axes that point in the directions of maximum variance.

2. **Projection**: Each data point in $X$ is projected onto this new coordinate system. For each row vector $\mathbf{x}_i$ (a point in $\mathbb{R}^n$):
   $$\mathbf{y}_i = \mathbf{x}_i V' = \begin{bmatrix} \mathbf{x}_i \cdot \mathbf{v}_1 & \mathbf{x}_i \cdot \mathbf{v}_2 & \cdots & \mathbf{x}_i \cdot \mathbf{v}_d \end{bmatrix}$$

3. **Dimensionality Reduction**: You're effectively taking each point in $n$-dimensional space and expressing it using only $d$ coordinates - the coordinates along the principal component directions. You're discarding the components along directions with low variance.

#### Intuitive View

Think of it as finding the "best viewing angle" for your data. If you have data in 3D that's mostly flat (like a pancake), PCA finds that the data lies approximately in a 2D plane. $V'$ defines that plane's orientation, and $Y$ gives you the coordinates of each point within that plane.


----

Returning to our example
$$X
= \begin{bmatrix}
2 & 1\\
-1 & 1\\
-1 & -2 \end{bmatrix},
\quad
S =
\begin{bmatrix}
3 & 1.5\\
1.5 & 3
\end{bmatrix}
$$

Eigenvalues of $S$: $\det(S - \lambda I) = (3-\lambda)^2 - 1.5^2 = 0$, so $\lambda_1 = 4.5, \: \lambda_2 = 1.5$.

Eigenvectors of $S$:
$$
\mathbf{v}_1 = \frac{1}{\sqrt{2}}
\begin{bmatrix}
1 \\ 1
\end{bmatrix}, \:
\mathbf{v}_2 = \frac{1}{\sqrt{2}}\begin{bmatrix}
1 \\ -1
\end{bmatrix}
$$

What is the value of the total variance in the data?

Total variance: $T = \lambda_1 + \lambda_2 = 6$.

This is also $S_{11} + S_{22} = \operatorname{trace}(S)$: the eigenvalues redistribute the per-feature variances across the principal directions without changing their sum. The first principal component accounts for $4.5/6 = 75\%$ of it.


---

Projecting onto the first principal component ($d = 1$):

$$
Y = XV' = X\mathbf{v}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix} 2 + 1 \\ -1 + 1 \\ -1 - 2\end{bmatrix} = \begin{bmatrix} 3/\sqrt{2} \\ 0 \\ -3/\sqrt{2}\end{bmatrix} \approx \begin{bmatrix} 2.12 \\ 0 \\ -2.12\end{bmatrix}.
$$

The variance of these three numbers is $\frac{1}{m-1}\left(\frac{9}{2} + 0 + \frac{9}{2}\right) = 4.5 = \lambda_1$, exactly as the derivation promised: the eigenvalue *is* the variance of the projected data.


---

Checking the arithmetic with numpy:

In [ ]:
import numpy as np

X0 = np.array([[12, 6],
               [9, 6],
               [9, 3]], dtype=float)
m = X0.shape[0]

X = X0 - X0.mean(axis=0)
S = X.T @ X / (m - 1)

eigenvalues, V = np.linalg.eigh(S)          # ascending order
order = np.argsort(eigenvalues)[::-1]
eigenvalues, V = eigenvalues[order], V[:, order]

Y = X @ V[:, :1]

print("S =\n", S)
print("eigenvalues:", eigenvalues, " explained variance:", eigenvalues / eigenvalues.sum())
print("v1 =", V[:, 0])
print("Y =", Y.ravel().round(4))
print("Var(Y1) =", Y.var(ddof=1).round(4), " lambda_1 =", eigenvalues[0].round(4))

## PCA Summary

- The columns of $V$ are the principal directions (or components).
- The reduced dimension data is the projection of the data in the direction of the principal directions, i.e.,  $Y=XV'$.
- The total variance $T$ in the data is the sum of all eigenvalues: $T = \lambda_1 + \lambda_2 + \dots + \lambda_n.$
- The first eigenvector $\mathbf{v}_1$ points in the most significant direction of the data. This direction explains the largest fraction $\lambda_1/T$ of the total variance.
- The second eigenvector $\mathbf{v}_2$ accounts for a smaller fraction $\lambda_2/T$.
- The **explained variance** of component $i$ is the value $\lambda_i/T$. These fractions are non-increasing in $i$ and sum to 1.
- The **cumulative (total) explained variance** is the sum of the explained variances up to component $k$, e.g. $\sum_{i=1}^{k} \frac{\lambda_i}{T}$. The total explained variance for all eigenvalues is 1.


## Case Study: Random Data

Let's consider some randomly generated data consisting of 2 features.

In [ ]:
#| fig-align: center
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from slideUtilities import OKABE_ITO

rng = np.random.default_rng(seed=42)

n_samples = 500
C = np.array([[0.1, 0.6], [2., .6]])
X0 = rng.standard_normal((n_samples, 2)) @ C + np.array([-6, 3])
X = X0 - X0.mean(axis=0)

plt.scatter(X[:, 0], X[:, 1])
plt.axis('equal')
plt.show()

---

Let's do PCA and plot the principal components over our dataset. As expected, the principal components are orthogonal and point in the directions of the maximum variance.

Each arrow is drawn with length $2\sqrt{\lambda_i}$, two standard deviations of the data along that component, so the arrows show the variance split as well as the directions.

In [ ]:
#| fig-align: center
origin = np.zeros(2)

# Compute the covariance matrix (X is already centered)
cov_matrix = np.cov(X, rowvar=False)

# Eigenvalues and eigenvectors, sorted in descending order
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]

plt.scatter(X[:, 0], X[:, 1], alpha=0.2)

for i, colour in enumerate([OKABE_ITO[3], OKABE_ITO[5]]):
    length = 2 * np.sqrt(eigenvalues[i])
    plt.quiver(*origin, *(length * eigenvectors[:, i]),
               angles='xy', scale_units='xy', scale=1, color=colour, width=0.008,
               label=f'PC{i+1}: $\\lambda_{i+1}$ = {eigenvalues[i]:.2f} '
                     f'({eigenvalues[i] / eigenvalues.sum():.0%} of variance)')

plt.legend(loc='upper left')
plt.title(r'Principal components, drawn with length $2\sqrt{\lambda_i}$')
plt.axis('equal')
plt.show()

The *sign* of each arrow is arbitrary: $\mathbf{v}$ and $-\mathbf{v}$ are both unit eigenvectors for the same $\lambda$, and `numpy.linalg.eigh`, `numpy.linalg.svd` and scikit-learn's `PCA` are each free to return either. Only the line through the origin and the variance along it are determined. Do not read meaning into a principal component pointing "left" rather than "right", and expect two libraries to disagree on signs.


## Case Study: Digits

Let's consider another example using scikit-learn's `digits` dataset: 1,797 handwritten digits scanned at $8\times 8$ pixels (from the UCI repository). It is a small cousin of the well-known MNIST set, whose images are $28\times 28$.

In [ ]:
#| fig-align: center
from sklearn import datasets
digits = datasets.load_digits()

plt.figure(figsize=(6, 6),)
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(digits.images[i], cmap='gray_r')
    plt.axis('off')
plt.tight_layout()
plt.show()

---

We first stack the columns of each digit on top of each other (this operation is called vectorizing) and perform PCA on the 64-D representation of the digits.

We can plot the explained variance ratio and the total (cumulative) explained variance ratio.

In [ ]:
#| fig-align: center
from sklearn.decomposition import PCA

X = digits.data
y = digits.target

import warnings
warnings.filterwarnings("ignore")

pca = PCA()
X_pca = pca.fit_transform(X)

# Create subplots
fig, axs = plt.subplots(2, 1, figsize=(6, 4))

# Scree plot (graph of eigenvalues corresponding to PC number)
# This shows the explained variance ratio
axs[0].plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_, marker='o', linestyle='--')
axs[0].set_title('Scree Plot')
axs[0].set_xlabel('Principal Component')
axs[0].set_ylabel('Explained Variance\n Ratio')
axs[0].grid(True)

# Cumulative explained variance plot
axs[1].plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
axs[1].set_title('Cumulative Explained Variance Plot')
axs[1].set_xlabel('Principal Component')
axs[1].set_ylabel('Cumulative Explained\n Variance')
axs[1].grid(True)

# Adjust layout
plt.tight_layout()
plt.show()

---

Let's plot the data in the first 2 principal component directions. We'll use the digit labels to color each digit in the reduced space.

In [ ]:
#| fig-align: center
# Okabe-Ito is a colour-blind-safe palette of eight colours; ten digits need two more.
DIGIT_CMAP = ListedColormap(OKABE_ITO + ['#999999', '#882255'])


def digit_scatter(ax, Z, labels, title, s=25):
    """Scatter a 2-D embedding Z, one colour per digit class."""
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap=DIGIT_CMAP, vmin=-0.5, vmax=9.5,
                    edgecolor='k', linewidth=0.3, s=s)
    ax.set_title(title)
    return sc


fig, ax = plt.subplots(figsize=(5, 5))
sc = digit_scatter(ax, X_pca, y, 'Digits in PC1 and PC2 Space', s=40)
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.legend(*sc.legend_elements(), title="Digit", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

---

We observe the following in our plot of the digits in the first two principal components:

- There is a decent clustering of some of our digits, in particular 0, 2, 3, 4, and 6.
- The numbers 0 and 6 seem to be relatively close to each other in this space.
- There is not a very clear separation of the number 5 from some of the other points.


## To scale or not to scale

Consider a situation where we have age (years) and height (feet) data for 4 people.

| Person | Age [years] | Height [feet] |
|--------|-------------|---------------|
| A      | 25          | 6.232         |
| B      | 30          | 6.232         |
| C      | 25          | 5.248         |
| D      | 30          | 5.248         |

![](figs/Scaling_feet.png)


Notice that the dominant direction of the variance is aligned horizontally.

---

What if the height is in cm?

| Person | Age [years] | Height [cm] |
|--------|-------------|-------------|
| A      | 25          | 189.95      |
| B      | 30          | 189.95      |
| C      | 25          | 159.96      |
| D      | 30          | 159.96      |

![](figs/Scaling_cm.png)


Notice that the dominant direction of the variance is aligned vertically.

---

Let's standardize our data: subtract each column's mean and divide by its standard deviation.

| Person | Age [years] | Height [cm] |
|--------|-------------|-------------|
| A      | -1          | 1           |
| B      | 1           | 1           |
| C      | -1          | -1          |
| D      | 1           | -1          |

![](figs/Scaling.png)


When we standardize our data, we observe an equal distribution of the variance.

The table divides by the *population* standard deviation (divide by $m$), which is what scikit-learn's `StandardScaler` does, so the entries are exactly $\pm 1$. Dividing by the sample standard deviation (divide by $m-1$, matching $S$) would give $\pm 0.87$ instead. Either way each column ends up with the same variance, which is all that matters here.


---

What quantity is represented by $\frac{Cov(X, Y)}{\sigma_X\sigma_Y}$, where $X$ and $Y$ represent the random variables of
a person's age and height, respectively?

This is the correlation coefficient, and the matrix of these values is the correlation matrix. When you _standardize_ your data you are no longer working with the covariance matrix but the correlation matrix.


PCA on a standardized dataset and working with the correlation matrix is the best practice when your data has very different scales.

PCA identifies the directions with the maximum variance and large scale data can disproportionately influence the results of your PCA.


When *not* to standardize: when the features already share a unit and their differing variances are the signal. Pixel intensities (the digits above), SNP counts (Novembre et al.), and last lecture's traffic counts are all left unscaled, because standardizing would inflate a nearly constant, noisy feature to the same weight as an informative one.



## Relationship to the SVD

Recall that the SVD of a (mean-centered) data matrix $X\in\mathbb{R}^{m\times n}$ ($m>n$) is

$$
X = U\Sigma V^T.
$$


In [ ]:
#| echo: false
#| fig-align: center
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Create a figure and axis
fig, ax = plt.subplots()

# Draw matrix A
rect_A = patches.Rectangle((0, 0), 2, 3, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_A)
ax.text(1, 1.5, r'$X$', fontsize=20, ha='center', va='center')
ax.text(1, -0.5, r'$(m \times n)$', fontsize=12, ha='center', va='center')

# Draw equal sign
ax.text(2.5, 1.5, r'$=$', fontsize=20, ha='center', va='center')

# Draw matrix U
rect_U = patches.Rectangle((3, 0), 2, 3, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_U)
ax.text(4, 1.5, r'$U$', fontsize=20, ha='center', va='center')
ax.text(4, -0.5, r'$(m \times n)$', fontsize=12, ha='center', va='center')

# Draw Sigma
rect_Sigma = patches.Rectangle((5.5, 1), 2, 2, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_Sigma)
ax.text(6.5, 2, r'$\Sigma$', fontsize=20, ha='center', va='center')
ax.text(6.5, 0.5, r'$(n \times n)$', fontsize=12, ha='center', va='center')

# Draw matrix V^T with the same dimensions as Sigma
rect_VT = patches.Rectangle((8, 1), 2, 2, linewidth=1, edgecolor='black', facecolor='none')
ax.add_patch(rect_VT)
ax.text(9, 2, r'$V^T$', fontsize=20, ha='center', va='center')
ax.text(9, 0.5, r'$(n \times n)$', fontsize=12, ha='center', va='center')

# Set limits and remove axes
ax.set_xlim(-1, 11)
ax.set_ylim(-2, 4)
ax.axis('off')

# Show the plot
plt.show()

---

How can we relate this to what we learned in PCA?

In PCA, we computed an eigen-decomposition of the covariance matrix, i.e., $S=V\Lambda V^{T}$.

Consider the following computation using the SVD of $X$

$$
\begin{align*}
X^{T}X
=
(U\Sigma V^T)^{T}(U\Sigma V^T)
=
V\Sigma U^TU \Sigma V^{T}
=
V\Sigma^{2}V^T
\end{align*}
$$

We see that eigenvalues $\lambda_i$ of the matrix $S$ are the squares $\sigma_{i}^{2}$ of the singular values $\Sigma$ (scaled by $\frac{1}{m-1}$).

## Scores, Reconstruction, and the Variance You Threw Away

Three identities tie PCA to last lecture's truncated SVD. Let $X = U\Sigma V^T$, let $V'$ be the first $d$ columns of $V$, $U'$ the first $d$ columns of $U$, and $\Sigma'$ the top-left $d\times d$ block of $\Sigma$.

- **Scores.** $Y = XV' = U\Sigma V^TV' = U'\Sigma'$, because $V^TV'$ is the first $d$ columns of the identity. Column $j$ of $Y$ is $\sigma_j\mathbf{u}_j$ and has variance $\sigma_j^2/(m-1) = \lambda_j$.
- **Reconstruction.** Mapping the scores back to $\mathbb{R}^n$, $X^{(d)} = YV'^T = U'\Sigma'V'^T$, is exactly the rank-$d$ truncated SVD. By Eckart--Young (lecture 12) it is the best rank-$d$ approximation of $X$ in the Frobenius norm. PCA with $d$ components and the rank-$d$ SVD of the *centered* data are the same thing.
- **Error.** $\|X - X^{(d)}\|_F^2 = \sum_{i>d}\sigma_i^2 = (m-1)\sum_{i>d}\lambda_i$. **The squared reconstruction error is the variance you threw away.** Dividing by $\|X\|_F^2 = (m-1)T$ turns it into $1 - (\text{cumulative explained variance through } d)$.


---

On the three-point example, $\sigma_1 = 3$ and $\sigma_2 = \sqrt{3}$ (since $\sigma_i^2 = (m-1)\lambda_i = 2\lambda_i$), and keeping one component should leave a squared error of $(m-1)\lambda_2 = 3$:

In [ ]:
X0 = np.array([[12, 6],
               [9, 6],
               [9, 3]], dtype=float)
m = X0.shape[0]
X = X0 - X0.mean(axis=0)

U, sigma, Vt = np.linalg.svd(X, full_matrices=False)
V = Vt.T

print("singular values:", sigma.round(4), "  sigma^2/(m-1):", (sigma**2 / (m - 1)).round(4))

Y = X @ V[:, :1]                      # scores, d = 1
print("X v1     :", Y.ravel().round(4), "  (sign flipped relative to eigh above: v1 vs -v1)")
print("u1 sigma1:", (U[:, 0] * sigma[0]).round(4), "  (identical, as Y = U' Sigma' says)")

X_d = Y @ V[:, :1].T                  # rank-1 reconstruction
print("||X - X_d||_F^2 =", np.sum((X - X_d)**2).round(4), "  (m-1) lambda_2 =", (sigma[1]**2).round(4))

---

In practice, PCA is done by computing the SVD of your data matrix as opposed to forming the covariance matrix and computing the eigenvalues.

The principal components are the right singular vectors $V$. The projected data is $Y = XV' = U'\Sigma'$. The eigenvalues of $S$ are obtained from squaring the entries of $\Sigma$ and scaling them by $\frac{1}{m-1}$.

Reasons to compute PCA this way are

- **Numerical Stability**: Forming $X^TX$ squares the condition number, $\kappa(X^TX) = \kappa(X)^2$ (lecture 11). Small singular values of $X$ become tiny eigenvalues of $S$ that can round to zero or even negative, and the corresponding eigenvectors are lost to noise. The SVD works on $X$ directly and never forms the product.

- **Efficiency**: SVD is computationally efficient, especially for large datasets. Many optimized algorithms and libraries (like those in NumPy and scikit-learn) leverage SVD for fast computations.

- **Direct Computation of Principal Components**: SVD directly provides the principal components (right singular vectors) and the singular values, which are related to the explained variance. This makes the process straightforward and avoids the need to compute the covariance matrix explicitly.


- **Memory Efficiency**: For large datasets, SVD can be more memory-efficient. Techniques like truncated SVD can be used to compute only the top principal components, reducing memory usage.

- **Versatility**: SVD is a fundamental linear algebra technique used in various applications beyond PCA, such as signal processing, image compression, and solving linear systems. This versatility makes it a well-studied and widely implemented method.


## Pros and Cons

Here are some advantages and disadvantages of using PCA.

**Advantages**

+ Allows for visualizations
+ Removes redundant variables
+ Can reduce overfitting (fewer inputs to the downstream model)
+ Speeds up other ML algorithms

**Disadvantages**

- reduces interpretability
- can result in information loss
- can be less effective than non-linear methods


# t-SNE

## When PCA Is Not Enough

PCA is linear: it chooses the best *flat* viewing angle. Some data has few intrinsic dimensions but is curled up in the ambient space, and no flat projection can unroll it.

In [ ]:
#| fig-align: center
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import TSNE

X_roll, t_roll = make_swiss_roll(n_samples=1500, noise=0.05, random_state=0)

roll_pca = PCA(n_components=2).fit_transform(X_roll)
roll_tsne = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X_roll)

fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(1, 3, 1, projection='3d')
ax.scatter(X_roll[:, 0], X_roll[:, 1], X_roll[:, 2], c=t_roll, cmap='viridis', s=8)
ax.view_init(elev=10, azim=-75)
ax.set_title('Swiss roll in 3-D')

ax = fig.add_subplot(1, 3, 2)
ax.scatter(roll_pca[:, 0], roll_pca[:, 1], c=t_roll, cmap='viridis', s=8)
ax.set_title('PCA to 2-D')

ax = fig.add_subplot(1, 3, 3)
ax.scatter(roll_tsne[:, 0], roll_tsne[:, 1], c=t_roll, cmap='viridis', s=8)
ax.set_title('t-SNE to 2-D')

for a in fig.axes[1:]:
    a.set_xticks([])
    a.set_yticks([])
plt.tight_layout()
plt.show()

Colour is the position along the roll. PCA squashes the roll flat, so successive turns land on top of each other and far-apart colours mix. t-SNE keeps each point next to its high-dimensional neighbours, so the colour varies smoothly across the map.

## What is t-SNE?

**t-SNE** stands for **t-distributed Stochastic Neighbor Embedding**

- A dimensionality reduction technique specifically designed for visualization
- Transforms high-dimensional data (hundreds or thousands of features) into 2D or 3D
- Goal: Preserve the local structure of your data while making it viewable
- Developed by Laurens van der Maaten and Geoffrey Hinton in 2008, see @van2008visualizing.

Think of it as creating a "map" of your data where similar points stay close together.

## Why Do We Need t-SNE?

**The Curse of Dimensionality**

- Real datasets often have many features (dimensions)
- Impossible to visualize data with 100+ dimensions
- Linear methods like PCA can only flatten, never unroll (previous slide)

**What t-SNE offers:**

- Reveals clusters and patterns in complex data
- Preserves neighborhood relationships between points
- Creates intuitive, interpretable visualizations

## How Does t-SNE Work?

### The Big Picture

**Step 1: Measure Similarity in High Dimensions**

- Calculate how "similar" each pair of points is in the original space
- Similar points get high probability, distant points get low probability

**Step 2: Create a Random Low-Dimensional Map**

- Start with random positions for all points in 2D/3D

**Step 3: Optimize the Map**

- Iteratively move points around to match the high-dimensional similarities
- Points that were neighbors in high-D should be neighbors in low-D

## Step 1: Compute Pairwise Similarities

- Calculate pairwise similarities between the $m$ points in the high-dimensional space.
- Use a Gaussian distribution to convert distances into probabilities.
- The probability $p_{j\vert i}$ between points $i$ and $j$ is given by:
  $$
  p_{j\vert i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}.
  $$

- The $p_{j\vert i}$ value represents the conditional probability that point $x_i$ would choose $x_j$ as its neighbor.
- Note that we set $p_{i\vert i} = 0$ -- we don't consider a point to be its own neighbor.

## Step 1: Symmetrize

- In general $p_{j\vert i} \neq p_{i\vert j}$: each point has its own bandwidth $\sigma_i$, and a point in a dense region "chooses" its neighbors differently from one in a sparse region.
- t-SNE therefore combines the two conditionals into a single joint probability
  $$
  p_{ij} = \frac{p_{j\vert i} + p_{i\vert j}}{2m},
  $$
  where $m$ is the number of data points.
- Dividing by $2m$ makes $\sum_{i \neq j} p_{ij} = 1$, so $P$ is one probability distribution over all pairs. It also guarantees $\sum_j p_{ij} > \frac{1}{2m}$ for every point $i$, so even an outlier contributes to the cost and is not simply ignored.
- Using this $p_{ij}$ is called symmetric t-SNE.

## Step 1: Choosing $\sigma$

- Every point gets its own bandwidth $\sigma_i$, chosen so that its Gaussian neighborhood contains a fixed *effective number of neighbors*. That number is the **perplexity**, the one hyperparameter you must set.
- Dense regions get a small $\sigma_i$, sparse regions a large one.
- Low perplexity (5--10) looks only at very local structure, and the map can fragment into many small clumps. High perplexity (50--100) uses broader neighborhoods, and small clusters merge. It must be smaller than the number of points.
- The perplexity commonly ranges between 5 and 50; scikit-learn's default is 30. Try several values and trust only the structure that persists -- we do exactly that on the digits below.


- The perplexity is defined as $\operatorname{Perp}(P_i) = 2^{H(P_i)}$, where $H(P_i)$ is the entropy of $P_i$. $P_i$ is the probability distribution induced by $\sigma_i$.
- The entropy $H(P_i) = -\sum_j p_{j\vert i} \log_2{(p_{j\vert i})}$.
- The value for $\sigma_i$ is found by binary search so that $\operatorname{Perp}(P_i)$ equals the chosen perplexity. If the $k$ nearest neighbors of $x_i$ were equally likely and everything else impossible, the perplexity would be exactly $k$ -- hence "effective number of neighbors".


## Step 2: Define Low-Dimensional Map

- Initialize points randomly in the low-dimensional space.
- Define a similar probability distribution using a simplified Student-t distribution:
  $$
  q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}.
  $$

## Step 3: Minimize Kullback-Leibler Divergence

- Minimize the Kullback-Leibler (KL) divergence between the high-dimensional and low-dimensional pair distributions:
  $$
  KL(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}.
  $$
- $KL(P\|Q) \geq 0$, with equality only when $Q = P$. A perfect map would reproduce every pairwise similarity, which is usually impossible in 2-D, so what matters is *which* mismatches cost the most. The weights $p_{ij}$ decide:
  - $p_{ij}$ large but $q_{ij}$ small (neighbors in high-D placed far apart in the map): a **large** penalty.
  - $p_{ij}$ small but $q_{ij}$ large (distant points in high-D placed close in the map): $p_{ij}\log(p_{ij}/q_{ij})$ stays **small**.
- So t-SNE works hard to keep neighbors together and barely cares where non-neighbors land relative to one another. This asymmetry is the reason local structure is preserved while cluster sizes and the gaps between clusters are not (see *Important Limitations*).
- To minimize the KL divergence we use gradient descent on $\partial KL / \partial y_i$ to iteratively adjust the positions of points, $y_i$, in the low-dimensional space.

## The Key Insight

**t-SNE uses different probability distributions for each step:**

- **High dimensions**: Gaussian (normal) distribution
  - Measures similarity based on Euclidean distance

- **Low dimensions**: Student's t-distribution with one degree of freedom
  - Has "heavier tails" than the Gaussian
  - Allows moderate distances in low-D to represent larger distances in high-D
  - Helps prevent crowding and creates better separation

This asymmetry is what makes t-SNE effective!

---

In [ ]:
#| fig-align: center
from scipy.stats import norm, t

x = np.linspace(-6, 6, 400)
gaussian_y = norm.pdf(x)        # mean 0, standard deviation 1
t_y = t.pdf(x, df=1)            # Student's t with one degree of freedom

plt.plot(x, gaussian_y, label='Gaussian (high-D similarity)')
plt.plot(x, t_y, linestyle='dashed', label="Student's t, 1 d.o.f. (low-D similarity)")

x0 = 3
plt.annotate(f'at a distance of {x0}:\nGaussian density {norm.pdf(x0):.4f}\n'
             f"Student's t density {t.pdf(x0, df=1):.3f}",
             xy=(x0, t.pdf(x0, df=1)), xytext=(3.2, 0.18), ha='center',
             arrowprops=dict(arrowstyle='->'))

plt.xlabel('distance between a pair of points')
plt.ylabel('probability density')
plt.title("Gaussian vs. Student's t-distribution")
plt.legend()
plt.show()

The two curves agree near zero, so close neighbors are treated alike. In the tails the Student's t keeps several times more mass: a pair that is moderately far apart in high-D can be placed *much* farther apart in the map and still match its (tiny) $p_{ij}$. That extra room is what relieves the **crowding problem** -- there is far less area in 2-D than volume in 64-D to spread the non-neighbors into.

## Case Study: Digits

Let's consider how t-SNE performs on the `digits` dataset we saw with PCA.

In [ ]:
#| fig-align: center
from sklearn.manifold import TSNE
from sklearn import datasets

digits = datasets.load_digits()
X = digits.data
y = digits.target

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(5, 5))
sc = digit_scatter(ax, X_tsne, y, 'Digits in t-SNE Space', s=40)
ax.set_xlabel('t-SNE Component 1')
ax.set_ylabel('t-SNE Component 2')
ax.legend(*sc.legend_elements(), title="Digit", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.show()

Compared with the PCA plot, every digit now forms its own island, including the 5s and 8s that PCA left tangled. Note that since scikit-learn 1.2 `TSNE` initializes the map from PCA (`init='pca'`) rather than at random, so repeated runs on the same data look alike; the classic "a different picture every run" behavior needs `init='random'`, which we use next.

## Perplexity and Seed Change the Picture

The same 1,797 digits, six times: three perplexities across, two random initializations down.

In [ ]:
#| fig-align: center
perplexities = [5, 30, 100]
seeds = [0, 1]

fig, axes = plt.subplots(len(seeds), len(perplexities), figsize=(10, 6.5))
for r, seed in enumerate(seeds):
    for c, perp in enumerate(perplexities):
        Z = TSNE(n_components=2, perplexity=perp, init='random',
                 random_state=seed).fit_transform(X)
        sc = digit_scatter(axes[r, c], Z, y, f'perplexity = {perp}, seed = {seed}', s=8)
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])

fig.legend(*sc.legend_elements(), title='Digit', loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.show()

- **Down a column** (same perplexity, different seed): the clusters are the same, but some of them move: at perplexity 5 and 100 whole digits jump to the other side of the map, and even at 30 the 0s trade places. Where a cluster lands, and which digit it sits next to, is partly an accident of the initialization.
- **Across a row** (same seed, different perplexity): at 5 each digit splinters into several loose fragments; at 100 the clusters are compact but 1, 8 and 9 start to bleed into a shared region. What survives every setting is *which points cluster together* -- that is the structure to trust, not where the clusters sit.


## Cluster Sizes and Gaps Are Not Preserved

Three Gaussian blobs in 10 dimensions, built so that we *know* the truth: two tight blobs (standard deviation 1) at a distance of 10 from each other, and one wide blob (standard deviation 5) another 30 away.

In [ ]:
#| fig-align: center
rng = np.random.default_rng(0)
n_per, dim = 200, 10

centres = np.zeros((3, dim))
centres[1, 0] = 10
centres[2, 0] = 40
stds = [1, 1, 5]

X_blobs = np.vstack([c + s * rng.standard_normal((n_per, dim)) for c, s in zip(centres, stds)])
blob_id = np.repeat([0, 1, 2], n_per)

blob_pca = PCA(n_components=2).fit_transform(X_blobs)
blob_tsne = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, Z, title in zip(axes, [blob_pca, blob_tsne], ['PCA', 't-SNE (perplexity 30)']):
    for k, (colour, std) in enumerate(zip(OKABE_ITO, stds)):
        mask = blob_id == k
        ax.scatter(Z[mask, 0], Z[mask, 1], s=10, color=colour, label=f'blob {k + 1}: std {std}')
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='datalim')
axes[0].legend(loc='upper left')
plt.tight_layout()
plt.show()

- **PCA is faithful.** The two tight blobs are tight and 10 apart; the wide blob is five times wider and 30 further on. Distances in the plot are distances in the data.
- **t-SNE equalizes.** All three blobs come out about the same size and roughly evenly spaced. The wide blob is shrunk and the big gap is closed -- exactly what the KL asymmetry predicts, since the cost barely notices where non-neighbors are placed.
- So from a t-SNE plot you may conclude "these points are neighbors", but not "this cluster is bigger" or "these two clusters are far apart".


## When to Use t-SNE

**Great for:**

- Exploratory data analysis
- Visualizing clusters in labeled data
- Understanding complex datasets (images, text, genomics)
- Finding patterns you didn't know existed

**Common applications:**

- Image datasets (like MNIST digits)
- Single-cell RNA sequencing data
- Word embeddings
- Customer segmentation

## Important Limitations


**1. It's for visualization only**

- There is no `transform` for new points, and the coordinates are not reusable features: do not feed a t-SNE map to a downstream model.
- Each run is a fresh optimization. With `init='random'` the picture changes with the seed (grid above); look for structure that is stable across seeds.

**2. Computationally expensive**

- Every pair of points enters the cost, so it is slow beyond roughly 10,000 points.
- Standardize, then PCA down to ~50 dimensions first, or subsample.


**3. Hyperparameters matter**

- Perplexity (neighborhood size) changes the picture (grid above). Typical range 5--50, default 30.
- Try 3--5 values and trust only what survives all of them.

**4. Sizes and distances are not meaningful**

- Cluster sizes are equalized and the gaps between clusters are rearranged (blobs above). This is the KL asymmetry from Step 3 made visible.
- Read local neighborhoods only, and use t-SNE alongside other analysis, never as the sole evidence.


## Recap: t-SNE vs PCA

| Feature | PCA | t-SNE |
|---------|-----|-------|
| **Speed** | Fast | Slow |
| **Purpose** | Dimensionality reduction | Visualization |
| **Preserves** | Global variance | Local neighborhoods |
| **Linear?** | Yes | No |
| **Deterministic?** | Yes | No (random initialization) |
| **Distances** | Meaningful | Not meaningful |
| **Further analysis?** | Yes | No |

**Best practice**: Use PCA first to reduce to ~50 dimensions, then apply t-SNE